In [1]:
import pandas as pd
import numpy as np
import glob
import os
from tqdm import tqdm

In [2]:
!pwd

/storage/Arushi/090526_EvoAge/kg_formation/processed_data_relation_wise_merge/generalised/OTHER_SPECIES/Mouse


In [3]:
BASE_DIR     = '/storage/Arushi/090526_EvoAge/kg_formation/'
MAPPING_DIR  = BASE_DIR + 'data_collection/databases_for_mapping/'
PROC_DIR     = BASE_DIR + 'processed_data/'

# ── Output path ───────────────────────────────────────────────────────────────
!mkdir Mouse_chemical_chemical
OUT_PATH = BASE_DIR + 'processed_data_relation_wise_merge/generalised/OTHER_SPECIES/Mouse/Mouse_chemical_chemical/Mouse_chemical_chemical.csv'

# ── Required output schema ────────────────────────────────────────────────────
REQUIRED_COLS = [
    'head', 'relation', 'tail',
    'head_type', 'relation_type', 'tail_type',
    'kg_source', 'kg_type',
    'head_id_is', 'tail_id_is',
    'head_detail_name', 'tail_detail_name', 'species'
]

In [4]:
#

# AgeAnnoMO

In [5]:
AgeAnnoMO = pd.read_csv(PROC_DIR + 'ageanno_processed_mo/AgeAnnoMO_Mouse_CHemical_Chemical_2.csv')
AgeAnnoMO.columns = AgeAnnoMO.columns.str.lower()
AgeAnnoMO = AgeAnnoMO[~AgeAnnoMO['head_detail_name'].isna()]

AgeAnnoMO['kg_type'] = 'Aging'
print(f"AgeAnnoMO: {len(AgeAnnoMO):,} rows")

AgeAnnoMO['species'] = 'Mus musculus'
AgeAnnoMO['kg_source'] = 'AgeAnnoMO'
AgeAnnoMO

AgeAnnoMO: 9,492 rows


,head,relation,tail,head_type,tail_type,head_detail_name,tail_detail_name,head_id_is,tail_id_is,relation_source,species,head_id,head_id_is,tail_id,tail_id_is,kg_type,kg_source
0,6083,ChemicalEntity_ChemicalEntity,28782,ChemicalEntity,ChemicalEntity,"[(2R,3S,4R,5R)-5-(6-aminopurin-9-yl)-3,4-dihyd...",(2S)-2-amino-5-(diaminomethylideneazaniumyl)pe...,Pubchem,Pubchem,AgeAnnoMO,Mus musculus,Adenosine monophosphate,Pubchem,(2S)-2-amino-5-(diaminomethylideneazaniumyl)pe...,Pubchem,Aging,AgeAnnoMO
1,28782,ChemicalEntity_ChemicalEntity,5287939,ChemicalEntity,ChemicalEntity,(2S)-2-amino-5-(diaminomethylideneazaniumyl)pe...,"[(3S,8S,9S,10R,13R,14S,17R)-10,13-dimethyl-17-...",Pubchem,Pubchem,AgeAnnoMO,Mus musculus,(2S)-2-amino-5-(diaminomethylideneazaniumyl)pe...,Pubchem,Cholesteryl linoleate,Pubchem,Aging,AgeAnnoMO
2,6083,ChemicalEntity_ChemicalEntity,6442676,ChemicalEntity,ChemicalEntity,"[(2R,3S,4R,5R)-5-(6-aminopurin-9-yl)-3,4-dihyd...","(Z)-N-[(2S,3R)-1,3-dihydroxyoctadecan-2-yl]oct...",Pubchem,Pubchem,AgeAnnoMO,Mus musculus,Adenosine monophosphate,Pubchem,N-(9Z-octadecenoyl)-sphinganine,Pubchem,Aging,AgeAnnoMO
3,6442676,ChemicalEntity_ChemicalEntity,5283566,ChemicalEntity,ChemicalEntity,"(Z)-N-[(2S,3R)-1,3-dihydroxyoctadecan-2-yl]oct...","N-[(E,2S,3R)-1,3-dihydroxyoctadec-4-en-2-yl]ic...",Pubchem,Pubchem,AgeAnnoMO,Mus musculus,N-(9Z-octadecenoyl)-sphinganine,Pubchem,N-icosanoylsphingosine,Pubchem,Aging,AgeAnnoMO
4,28782,ChemicalEntity_ChemicalEntity,102615,ChemicalEntity,ChemicalEntity,(2S)-2-amino-5-(diaminomethylideneazaniumyl)pe...,(3-hydroxy-2-octadecanoyloxypropyl) octadecanoate,Pubchem,Pubchem,AgeAnnoMO,Mus musculus,(2S)-2-amino-5-(diaminomethylideneazaniumyl)pe...,Pubchem,"1,2-Distearoyl-rac-glycerol",Pubchem,Aging,AgeAnnoMO
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9487,439918,ChemicalEntity_ChemicalEntity,9934,ChemicalEntity,ChemicalEntity,(2S)-2-(diaminomethylideneamino)butanedioic acid,"1,3-thiazolidine-4-carboxylic acid",Pubchem,Pubchem,AgeAnnoMO,Mus musculus,MFCD00055819,Pubchem,Timonacic,Pubchem,Aging,AgeAnnoMO
9488,866,ChemicalEntity_ChemicalEntity,123372,ChemicalEntity,ChemicalEntity,"2,6-diaminohexanoic acid",ethyl 4-isothiocyanatobutanoate,Pubchem,Pubchem,AgeAnnoMO,Mus musculus,DL-Lysine,Pubchem,Ethyl 4-isothiocyanatobutanoate,Pubchem,Aging,AgeAnnoMO
9489,849,ChemicalEntity_ChemicalEntity,123372,ChemicalEntity,ChemicalEntity,piperidine-2-carboxylic acid,ethyl 4-isothiocyanatobutanoate,Pubchem,Pubchem,AgeAnnoMO,Mus musculus,Pipecolic acid,Pubchem,Ethyl 4-isothiocyanatobutanoate,Pubchem,Aging,AgeAnnoMO
9490,111306,ChemicalEntity_ChemicalEntity,123372,ChemicalEntity,ChemicalEntity,(2S)-pyrrolidine-2-carboxamide,ethyl 4-isothiocyanatobutanoate,Pubchem,Pubchem,AgeAnnoMO,Mus musculus,Prolinamide,Pubchem,Ethyl 4-isothiocyanatobutanoate,Pubchem,Aging,AgeAnnoMO


# Consolidate into Unified Schem

In [6]:
# List all source DataFrames to include
source_dfs = [
    AgeAnnoMO, 
    
]

aligned = []
for df in source_dfs:
    df = df.copy()
    for col in REQUIRED_COLS:
        if col not in df.columns:
            df[col] = None       
    aligned.append(df[REQUIRED_COLS])

final_df = pd.concat(aligned, ignore_index=True)
print(f"Consolidated rows: {len(final_df):,}")
final_df = final_df.loc[:, ~final_df.columns.duplicated()]
final_df

Consolidated rows: 9,492


,head,relation,tail,head_type,relation_type,tail_type,kg_source,kg_type,head_id_is,tail_id_is,head_detail_name,tail_detail_name,species
0,6083,ChemicalEntity_ChemicalEntity,28782,ChemicalEntity,None,ChemicalEntity,AgeAnnoMO,Aging,Pubchem,Pubchem,"[(2R,3S,4R,5R)-5-(6-aminopurin-9-yl)-3,4-dihyd...",(2S)-2-amino-5-(diaminomethylideneazaniumyl)pe...,Mus musculus
1,28782,ChemicalEntity_ChemicalEntity,5287939,ChemicalEntity,None,ChemicalEntity,AgeAnnoMO,Aging,Pubchem,Pubchem,(2S)-2-amino-5-(diaminomethylideneazaniumyl)pe...,"[(3S,8S,9S,10R,13R,14S,17R)-10,13-dimethyl-17-...",Mus musculus
2,6083,ChemicalEntity_ChemicalEntity,6442676,ChemicalEntity,None,ChemicalEntity,AgeAnnoMO,Aging,Pubchem,Pubchem,"[(2R,3S,4R,5R)-5-(6-aminopurin-9-yl)-3,4-dihyd...","(Z)-N-[(2S,3R)-1,3-dihydroxyoctadecan-2-yl]oct...",Mus musculus
3,6442676,ChemicalEntity_ChemicalEntity,5283566,ChemicalEntity,None,ChemicalEntity,AgeAnnoMO,Aging,Pubchem,Pubchem,"(Z)-N-[(2S,3R)-1,3-dihydroxyoctadecan-2-yl]oct...","N-[(E,2S,3R)-1,3-dihydroxyoctadec-4-en-2-yl]ic...",Mus musculus
4,28782,ChemicalEntity_ChemicalEntity,102615,ChemicalEntity,None,ChemicalEntity,AgeAnnoMO,Aging,Pubchem,Pubchem,(2S)-2-amino-5-(diaminomethylideneazaniumyl)pe...,(3-hydroxy-2-octadecanoyloxypropyl) octadecanoate,Mus musculus
...,...,...,...,...,...,...,...,...,...,...,...,...,...
9487,439918,ChemicalEntity_ChemicalEntity,9934,ChemicalEntity,None,ChemicalEntity,AgeAnnoMO,Aging,Pubchem,Pubchem,(2S)-2-(diaminomethylideneamino)butanedioic acid,"1,3-thiazolidine-4-carboxylic acid",Mus musculus
9488,866,ChemicalEntity_ChemicalEntity,123372,ChemicalEntity,None,ChemicalEntity,AgeAnnoMO,Aging,Pubchem,Pubchem,"2,6-diaminohexanoic acid",ethyl 4-isothiocyanatobutanoate,Mus musculus
9489,849,ChemicalEntity_ChemicalEntity,123372,ChemicalEntity,None,ChemicalEntity,AgeAnnoMO,Aging,Pubchem,Pubchem,piperidine-2-carboxylic acid,ethyl 4-isothiocyanatobutanoate,Mus musculus
9490,111306,ChemicalEntity_ChemicalEntity,123372,ChemicalEntity,None,ChemicalEntity,AgeAnnoMO,Aging,Pubchem,Pubchem,(2S)-pyrrolidine-2-carboxamide,ethyl 4-isothiocyanatobutanoate,Mus musculus


# Sanity Check — Distinct Values

In [7]:
for col in ['relation', 'head_type', 'tail_type', 'relation_type', 'kg_source', 'head_id_is', 'tail_id_is']:
    print(f"{col:20s}: {set(final_df[col])}")

relation            : {'ChemicalEntity_ChemicalEntity'}
head_type           : {'ChemicalEntity'}
tail_type           : {'ChemicalEntity'}
relation_type       : {None}
kg_source           : {'AgeAnnoMO'}
head_id_is          : {'Pubchem'}
tail_id_is          : {'Pubchem'}


In [8]:
# Step 4: drop unresolvable rows
before = len(final_df)
final_df = final_df[~final_df['tail_detail_name'].isna()].reset_index(drop=True)
print(f"Dropped {before - len(final_df):,} unresolvable rows → {len(final_df):,} remaining")

Dropped 0 unresolvable rows → 9,492 remaining


# NaN Audit (pre-dedup)

In [9]:
true_nan   = final_df.isna().sum()
string_nan = final_df.apply(lambda col: col.astype(str).str.upper().eq('NAN').sum())

pd.DataFrame({
    'NaN_count':          true_nan,
    "'NAN'_string_count": string_nan,
    'Total_NaN_like':     true_nan + string_nan
})

,NaN_count,'NAN'_string_count,Total_NaN_like
head,0,0,0
relation,0,0,0
tail,0,0,0
head_type,0,0,0
relation_type,9492,0,9492
tail_type,0,0,0
kg_source,0,0,0
kg_type,0,0,0
head_id_is,0,0,0
tail_id_is,0,0,0


# Deduplication

In [10]:
def merge_sources(x):
    """Combine unique, non-null source labels with '::' delimiter."""
    return '::'.join(sorted(set(x.dropna())))

group_cols = ['head', 'relation', 'tail']

final_df_dedup = final_df.groupby(group_cols, as_index=False).agg({
    'head_type':        'first',
    'relation_type':    'first',
    'tail_type':        'first',
    'kg_source':        merge_sources,
    'kg_type':          merge_sources,   # ← changed from 'first'
    'head_id_is':       'first',
    'tail_id_is':       'first',
    'head_detail_name': 'first',
    'tail_detail_name': 'first',
    'species': 'first'
})

print(f"Before dedup: {len(final_df):,}  |  After dedup: {len(final_df_dedup):,}")
final_df_dedup.head(3)

Before dedup: 9,492  |  After dedup: 9,370


,head,relation,tail,head_type,relation_type,tail_type,kg_source,kg_type,head_id_is,tail_id_is,head_detail_name,tail_detail_name,species
0,288,ChemicalEntity_ChemicalEntity,792,ChemicalEntity,None,ChemicalEntity,AgeAnnoMO,Aging,Pubchem,Pubchem,3-hydroxy-4-(trimethylazaniumyl)butanoate,[3-(1H-imidazol-5-yl)-2-oxopropyl] dihydrogen ...,Mus musculus
1,288,ChemicalEntity_ChemicalEntity,8160,ChemicalEntity,None,ChemicalEntity,AgeAnnoMO,Aging,Pubchem,Pubchem,3-hydroxy-4-(trimethylazaniumyl)butanoate,2-butoxyethyl acetate,Mus musculus
2,288,ChemicalEntity_ChemicalEntity,8923,ChemicalEntity,None,ChemicalEntity,AgeAnnoMO,Aging,Pubchem,Pubchem,3-hydroxy-4-(trimethylazaniumyl)butanoate,2-[2-(2-butoxyethoxy)ethoxy]ethanol,Mus musculus


In [11]:
true_nan   = final_df_dedup.isna().sum()
string_nan = final_df_dedup.apply(lambda col: col.astype(str).str.upper().eq('NAN').sum())

pd.DataFrame({
    'NaN_count':          true_nan,
    "'NAN'_string_count": string_nan,
    'Total_NaN_like':     true_nan + string_nan
})

,NaN_count,'NAN'_string_count,Total_NaN_like
head,0,0,0
relation,0,0,0
tail,0,0,0
head_type,0,0,0
relation_type,9370,0,9370
tail_type,0,0,0
kg_source,0,0,0
kg_type,0,0,0
head_id_is,0,0,0
tail_id_is,0,0,0


In [12]:
print("kg_source values present:", set(final_df_dedup['kg_source']), final_df_dedup['kg_source'].value_counts())

kg_source values present: {'AgeAnnoMO'} kg_source
AgeAnnoMO    9370
Name: count, dtype: int64


In [13]:
print("kg_source values present:", set(final_df_dedup['kg_type']), final_df_dedup['kg_type'].value_counts())

kg_source values present: {'Aging'} kg_type
Aging    9370
Name: count, dtype: int64


In [14]:
! mkdir Mouse_chemical_chemical

mkdir: cannot create directory ‘Mouse_chemical_chemical’: File exists


In [15]:
final_df_dedup.to_csv(OUT_PATH, index=False)
print(f"Saved {len(final_df_dedup):,} rows → {OUT_PATH}")

Saved 9,370 rows → /storage/Arushi/090526_EvoAge/kg_formation/processed_data_relation_wise_merge/generalised/OTHER_SPECIES/Mouse/Mouse_chemical_chemical/Mouse_chemical_chemical.csv


In [16]:
# OUT_PATH